# Module 07: TensorFlow & Keras for Deep Learning
## Notebook 04: High-Performance Data Pipelines with `tf.data`

Training deep learning models on modern accelerators (GPUs/TPUs) requires saturating the compute pipeline. If data extraction and preprocessing bottleneck the CPU, the accelerator sits idle (GPU starvation). The **`tf.data`** API builds scalable, multi-threaded input pipelines that overlap CPU preprocessing with accelerator execution.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Understand the **ETL (Extract, Transform, Load)** paradigm in deep learning input pipelines.
2. Construct datasets from memory slices with `tf.data.Dataset.from_tensor_slices()`.
3. Apply transformations: `.shuffle()`, `.batch()`, `.map()`, `.filter()`, and `.repeat()`.
4. Maximize pipeline throughput using **`.prefetch(tf.data.AUTOTUNE)`** and parallel map operations.
5. **Advanced:** Construct an **End-to-End Tabular Feature Processing Pipeline** using Keras Preprocessing Layers (`Normalization`, `StringLookup`, `CategoryEncoding`) directly inside the model graph to permanently prevent training/serving skew.

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
import keras
from keras import layers, models
import numpy as np
import pandas as pd

# Dynamic path resolution for shared curriculum datasets
data_dir = "data_files" if os.path.exists("data_files") else "../data_files"
print(f"TensorFlow Version: {tf.__version__}")
print(f"Data Directory: {os.path.abspath(data_dir)}")

TensorFlow Version: 2.21.0
Data Directory: /home/admin/code/notebooks/data_files


### 1. Dataset Construction & Transformations
The core abstraction is `tf.data.Dataset`:
- `from_tensor_slices((X, y))`: Creates a dataset from in-memory NumPy arrays or tensors.
- `.shuffle(buffer_size)`: Randomly shuffles samples using a memory buffer.
- `.batch(batch_size)`: Combines consecutive elements into batches.
- `.map(map_func)`: Applies element-wise transformations (e.g. normalization, augmentations).

In [4]:
# 1. Create dataset from tensor slices
features = tf.random.normal([1000, 8])
labels = tf.random.uniform([1000, 1], minval=0, maxval=2, dtype=tf.int32)

dataset = tf.data.Dataset.from_tensor_slices((features, labels))
print(f"Single element spec: {dataset.element_spec}")

# 2. Chain pipeline transformations
batch_size = 32
pipeline = (
    dataset
    .shuffle(buffer_size=500, seed=42)
    .batch(batch_size)
)

# Inspect first batch
for batch_x, batch_y in pipeline.take(1):
    print(f"\nBatch X shape: {batch_x.shape}, Batch y shape: {batch_y.shape}")

Single element spec: (TensorSpec(shape=(8,), dtype=tf.float32, name=None), TensorSpec(shape=(1,), dtype=tf.int32, name=None))

Batch X shape: (32, 8), Batch y shape: (32, 1)


### 2. High-Throughput Performance Engineering: Prefetching & Parallelism
Naive pipelines execute sequentially: CPU loads batch $\to$ GPU trains on batch $\to$ CPU loads next batch.
To achieve maximum throughput:
- **`prefetch(tf.data.AUTOTUNE)`:** Decouples data production from consumption. While the GPU is training on step $k$, the CPU is already preparing step $k+1$.
- **`map(num_parallel_calls=tf.data.AUTOTUNE)`:** Distributes CPU-intensive transformations across all available CPU cores.

In [5]:
def augment_and_scale(x, y):
    # Element-wise preprocessing transformation
    x_scaled = x * 1.5 + 0.1
    return x_scaled, y

# Production high-throughput pipeline configuration
optimized_pipeline = (
    dataset
    .shuffle(buffer_size=1000)
    .map(augment_and_scale, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(64)
    .prefetch(buffer_size=tf.data.AUTOTUNE) # Automatically tunes buffer depth
)

print(f"Optimized Pipeline: {optimized_pipeline}")

Optimized Pipeline: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 8), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.int32, name=None))>


### 3. Complex Application: End-to-End Tabular Feature Processing with Keras Layers
Traditional ML applies preprocessing (scaling, one-hot encoding) in external scripts (e.g. Scikit-Learn). At deployment time, production software must duplicate these steps, causing **Training/Serving Skew**.
**Best Practice:** Embed preprocessing layers directly into the Keras model:
- `layers.Normalization()`: Adapts to feature mean and variance.
- `layers.StringLookup()`: Builds vocabulary indices for strings.
- `layers.CategoryEncoding()`: Encodes category indices into one-hot or multi-hot vectors.
Below, we load `data_files/housing_market.csv` and build an end-to-end model that accepts raw strings and raw floats directly!

In [7]:
# Load tabular dataset from data_files
csv_path = os.path.join(data_dir, "housing_market.csv")
df = pd.read_csv(csv_path)
print(f"Loaded {csv_path}: {df.shape[0]} rows, columns: {list(df.columns)}")

# Split features and target
target_col = "Price"
features_df = df.copy()
targets = features_df.pop(target_col).values.astype(np.float32)

# Exclude non-feature identifier column if present
features_df.drop(columns=["House_ID"], inplace=True, errors="ignore")

# Define numeric and categorical feature columns matching dataset schema
numeric_cols = [c for c in ["SquareFeet", "Square_Feet", "Bedrooms", "Age_Years"] if c in features_df.columns]
cat_cols = [c for c in ["Location", "Neighborhood"] if c in features_df.columns]
print(f"Numeric feature columns: {numeric_cols}")
print(f"Categorical feature columns: {cat_cols}")

# Convert pandas dataframe into tf.data.Dataset yielding 2D feature tensors (batch_size, 1)
def df_to_dataset(dataframe, labels, batch_size=16):
    df_dict = {}
    for col in dataframe.columns:
        # Reshape to (N, 1) so dataset yields (batch_size, 1) matching layers.Input(shape=(1,))
        arr = np.array(dataframe[col].values)[:, np.newaxis]
        if col in numeric_cols:
            arr = arr.astype(np.float32)
        df_dict[col] = arr
    ds = tf.data.Dataset.from_tensor_slices((df_dict, labels))
    return ds.shuffle(buffer_size=len(dataframe)).batch(batch_size)

train_ds = df_to_dataset(features_df, targets)

# 1. Numeric Feature Preprocessing: Normalization layers
num_inputs = []
num_encoded = []
for col in numeric_cols:
    inp = layers.Input(shape=(1,), name=col, dtype=tf.float32)
    norm = layers.Normalization(axis=-1)
    # Adapt normalization layer on 2D array of shape (N, 1) so mean/variance have shape (1,)
    norm.adapt(features_df[[col]].values.astype(np.float32))
    num_inputs.append(inp)
    num_encoded.append(norm(inp))

# 2. Categorical Feature Preprocessing: StringLookup + CategoryEncoding
cat_inputs = []
cat_encoded = []
for col in cat_cols:
    inp = layers.Input(shape=(1,), name=col, dtype=tf.string)
    lookup = layers.StringLookup(output_mode="one_hot")
    lookup.adapt(features_df[[col]].values)
    cat_inputs.append(inp)
    encoded = lookup(inp)
    # Ensure 2D tensor of shape (batch_size, vocab_size) for concatenation
    encoded = layers.Flatten()(encoded)
    cat_encoded.append(encoded)

# 3. Concatenate all encoded feature branches
all_features = layers.concatenate(num_encoded + cat_encoded)
x = layers.Dense(32, activation="relu")(all_features)
x = layers.Dense(16, activation="relu")(x)
output = layers.Dense(1, activation="linear", name="price_prediction")(x)

# Build End-to-End Model accepting RAW inputs
end_to_end_model = models.Model(
    inputs=num_inputs + cat_inputs,
    outputs=output,
    name="End_to_End_Housing_Regressor"
)
end_to_end_model.compile(optimizer="adam", loss="mse", metrics=["mae"])

# Train directly on raw tabular dataset
end_to_end_model.fit(train_ds, epochs=10, verbose=1)

# Verify inference with a raw dictionary sample using tf.constant (ensures canonical tf.string & tf.float32)
raw_sample = {}
for col in numeric_cols:
    if "Square" in col:
        raw_sample[col] = tf.constant([[2200.0]], dtype=tf.float32)
    elif col == "Bedrooms":
        raw_sample[col] = tf.constant([[3.0]], dtype=tf.float32)
    elif col == "Age_Years":
        raw_sample[col] = tf.constant([[12.0]], dtype=tf.float32)

for col in cat_cols:
    sample_val = "Suburban" if col == "Location" else "Downtown"
    raw_sample[col] = tf.constant([[sample_val]], dtype=tf.string)

pred_price = end_to_end_model.predict(raw_sample, verbose=0)
print(f"\nInference Prediction for Raw Input: ${float(pred_price[0, 0]):,.2f}")

Loaded ../data_files/housing_market.csv: 50 rows, columns: ['House_ID', 'Square_Feet', 'Bedrooms', 'Location', 'Price']
Numeric feature columns: ['Square_Feet', 'Bedrooms']
Categorical feature columns: ['Location']
Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 328649801728.0000 - mae: 542961.8125  
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 328649703424.0000 - mae: 542961.7500
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 328649605120.0000 - mae: 542961.6250
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 328649506816.0000 - mae: 542961.5000
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 328649375744.0000 - mae: 542961.3750
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 328649277440.0000 - mae: 542961.3125
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 328649146368.0000 - mae: 542961.1875
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 328649015296.0000 - mae: 542961.0625
Epoch 9/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 